# TCC — Ambiente inicial local

Execução a partir dos arquivos do repositório, com Python 3.10+.


## 1. Localizar o repositório


In [ ]:
from pathlib import Path
import sys

# Descobre a raiz a partir da pasta local, sem caminhos pessoais fixos.
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "TCC_Codigos/data/project_graph.json").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Abra o projeto a partir da raiz do repositório TCC.")
PROJECT_DIR = REPO_ROOT / "TCC_Codigos"
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Projeto local:", PROJECT_DIR)


## 2. Confirmar a pasta local do projeto


In [ ]:
print("Pasta configurada:", PROJECT_DIR)
assert PROJECT_DIR.is_dir()


## 3. Verificar arquivos essenciais

In [ ]:

required_files = [
    PROJECT_DIR / "README.md",
    PROJECT_DIR / "src",
    PROJECT_DIR / "data",
    PROJECT_DIR / "tests",
]

for item in required_files:
    status = "OK" if item.exists() else "FALTANDO"
    print(f"{status:10} -> {item}")



## 4. Adicionar a pasta do projeto ao Python

Isso permite importar os módulos que estão dentro de `src`.


In [ ]:

import sys

src_path = str(PROJECT_DIR / "src")

if src_path not in sys.path:
    sys.path.append(src_path)

print("src adicionada ao Python:")
print(src_path)


## 5. Executar os testes automatizados

In [ ]:
import subprocess

subprocess.run([sys.executable, str(PROJECT_DIR / "scripts/test_offline.py")], check=True)



Se os testes passarem, significa que a estrutura básica do projeto está funcionando.

O objetivo aqui não é ainda validar a hipótese científica, mas confirmar que o código consegue executar sem erro.


## 6. Carregar o grafo do Projeto Alfa

In [ ]:

import json

graph_file = PROJECT_DIR / "data" / "project_graph.json"

with open(graph_file, "r", encoding="utf-8") as f:
    project_graph = json.load(f)

print("Nome do projeto:", project_graph.get("name"))
print("Número de entidades:", len(project_graph.get("nodes", [])))
print("Número de relações:", len(project_graph.get("edges", [])))


## 7. Visualizar algumas entidades

In [ ]:

for node in project_graph["nodes"][:10]:
    print(node)


## 8. Visualizar algumas relações

In [ ]:

for edge in project_graph["edges"][:15]:
    print(edge)



## 9. Testar um exemplo de propagação de impacto

Esta célula tenta usar o motor de impacto do pacote.

Caso o nome do módulo seja diferente na versão atual do projeto, a própria célula mostrará a mensagem de ajuste necessária.


In [ ]:
from tcc_agent.graph import ProjectGraph

graph_file = PROJECT_DIR / "data" / "project_graph.json"

project = ProjectGraph.from_json(graph_file)

print("Projeto carregado.")
print("Entidades:", len(project.nodes))
print("Relações:", len(project.edges))

In [ ]:
impactados, caminhos = project.downstream(["SUP_MOTOR"])

print("Entidades potencialmente impactadas:\n")

for entity_id in impactados:
    nome = project.entity_name(entity_id)
    print(f"- {entity_id}: {nome}")

print("\nCaminhos de propagação:\n")

for entity_id, caminho in caminhos.items():
    nomes = [project.entity_name(x) for x in caminho]
    print(" -> ".join(nomes))

## 10. Carregar os 36 cenários experimentais

In [ ]:

scenarios_file = PROJECT_DIR / "data" / "scenarios_ground_truth.json"

with open(scenarios_file, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

print("Quantidade de cenários:", len(scenarios))


## 11. Exibir o primeiro cenário

In [ ]:

from pprint import pprint

pprint(scenarios[0])


## 12. Listar ID e texto dos 36 cenários

In [ ]:

for scenario in scenarios:
    print(f"{scenario['scenario_id']}: {scenario['source_text']}")



## 13. Separar casos para o piloto do baseline

Sugestão de seis casos iniciais:

- S001
- S003
- S007
- S009
- S024
- S035

Esses casos incluem eventos simples, difíceis e casos negativos.


In [ ]:

pilot_ids = {"S001", "S003", "S007", "S009", "S024", "S035"}

pilot_scenarios = [
    s for s in scenarios
    if s["scenario_id"] in pilot_ids
]

for s in pilot_scenarios:
    print("\n", s["scenario_id"])
    print(s["source_text"])



## 14. Próxima etapa: integrar um LLM

Ainda não colocamos a chave de API neste notebook.

Isso é proposital.

Antes, vocês devem decidir:

1. qual modelo será usado;
2. qual versão;
3. qual temperatura;
4. qual prompt será congelado;
5. como as respostas serão registradas.

Depois dessa decisão, criem um novo notebook:

`02_baseline_llm.ipynb`

Esse notebook deverá executar os 36 cenários e salvar as respostas em arquivo, sem alterar o ground truth.
